# 분석한 댓글 감성분석를 점수로 환산하는 노트북입니다.

In [10]:
from dotenv import load_dotenv
import db_utils
import torch
from importlib import reload
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sqlalchemy import text
import pandas as pd

In [4]:
db_utils.test_connection()

DB 연결 성공
host     : 127.0.0.1:3306
database : youtube_model_db


In [12]:
load_dotenv("../../.env", override=True)

query = """
    SELECT 
        cit.youtube_channel_id as channel_id,
        COUNT(CASE WHEN ct.sentiment_label = 1 THEN 1 END) AS sentiment_1_count,
        COUNT(CASE WHEN ct.sentiment_label = 0 THEN 1 END) AS sentiment_0_count
    FROM comments_table ct 
    RIGHT JOIN channel_info_table cit 
    ON ct.channel_identifier = cit.channel_identifier
    WHERE ct.sentiment_label IS NOT NULL
    GROUP BY cit.youtube_channel_id 
"""
comments_df = db_utils.run_query(query, display=True)

print(f"불러온 데이터 개수: {len(comments_df)}")


,channel_id,sentiment_1_count,sentiment_0_count
0,UCSshlhZx6y5hAyzPxxuSO6Q,351,146
1,UCLLlKJZVbeUTzwjVM7Lx9kg,309,140
2,UCsaQlcW1DX-UYpozEwBUDzQ,117,119
3,UChXOKeqCxhM3AjyNb7rAAOw,146,226
4,UC7xrN8sK99nXgE_wL-3f7Kg,189,149
...,...,...,...
221,UCGEnF364gHB5y6me8eA4gsQ,6,17
222,UCS1bJuQ8_1KXC86RhPvJexw,30,27
223,UCCcK3JRvJWPvOfw2nKNmIIg,6,9
224,UClSxH8KQdfytf2raNAByMiA,6,7


불러온 데이터 개수: 226


In [11]:
dataset_wide_df = pd.read_csv("/Users/sj.kang/Desktop/project2/SKN30-2nd-1Team/data/raw/filtered_dataset_wide.csv")
dataset_wide_df.head()

,channel_id,title,published_at,subscriber_count,view_count,total_video_count,source_file,collected_video_count,days_since_last_upload,avg_upload_interval_days,...,hiatus_count_30d,upload_freq_change_rate,avg_view_count,std_view_count,avg_like_count,avg_comment_count,avg_engagement_rate,shorts_ratio,avg_shorts_view,avg_normal_view
0,UC0_yHLyKYYvG4bzylDs3pwA,2M (투엠) ENT,2009-03-16T14:52:53Z,100000,10030245,557,channels_must_3250_3299.csv,50.0,2.0,0.96,...,0.0,5.000000e+10,3583.14,8651.12,84.52,2.96,0.035743,0.96,3687.46,1079.50
1,UCDYdwDXhbVZXkdV6kZn-0IQ,심킹 (ABYSSKIMG),2012-05-01T15:10:54Z,100000,9668787,112,channels_must_3250_3299.csv,50.0,1120.0,22.61,...,5.0,0.000000e+00,55364.56,57049.86,483.32,176.76,0.014234,0.06,1549.67,58799.55
2,UCbr6AL4ayehAkXcdBM68e8A,김PD전당포,2012-01-06T08:48:32Z,100000,2156709,135,channels_must_3250_3299.csv,50.0,448.0,36.69,...,7.0,0.000000e+00,1516.14,2263.11,34.86,2.16,0.015272,0.32,2111.50,1235.97
3,UCJQDi71H00IXCQbxAtFem3Q,DC튜브,2015-12-19T23:24:19Z,100000,30385528,2043,channels_must_3250_3299.csv,50.0,14.0,14.12,...,10.0,5.000000e-01,5020.40,4387.90,62.90,21.66,0.019500,0.58,4605.66,5593.14
4,UCR33sqrqf6rlT2AlpgyKmyA,두리 원투쓰리코,2019-03-12T15:59:11Z,109000,123170488,1630,channels_must_3250_3299.csv,50.0,1.0,0.00,...,0.0,5.000000e+10,20873.78,57756.46,88.86,6.80,0.007655,0.96,21373.56,8879.00


In [ ]:
merged_df = pd.merge(dataset_wide_df, comments_df, on ="channel_id", how ="right")

(226, 24)

In [20]:
merged_df_sorted = merged_df.sort_values(by = "sentiment_1_count", ascending=False)


In [24]:
merged_df_sorted_bad = merged_df.sort_values(by = "sentiment_0_count", ascending=False)

In [25]:
merged_df_sorted_bad

,channel_id,title,published_at,subscriber_count,view_count,total_video_count,source_file,collected_video_count,days_since_last_upload,avg_upload_interval_days,...,avg_view_count,std_view_count,avg_like_count,avg_comment_count,avg_engagement_rate,shorts_ratio,avg_shorts_view,avg_normal_view,sentiment_1_count,sentiment_0_count
48,UCdK23vzK74N6SzL9VIKR3MQ,소피아패밀리(Sophia family),2017-10-21T15:41:23Z,401000,223620148,313,channels_must_5900_5949.csv,50.0,5.0,3.80,...,389615.82,244711.27,17071.28,899.84,0.051985,0.38,464340.37,343816.90,168,332
152,UC5m1d4sPHnpcLCJEu3j2FpQ,최민준의 아들TV,2015-10-22T00:13:49Z,953000,136109092,431,channels_must_7150_7199.csv,50.0,3.0,5.61,...,178043.88,139248.39,3297.02,148.80,0.019659,0.00,NaN,178043.88,156,329
151,UCeXxAet11DPbC8pG-lBOUTQ,아리둥절 Ari the Corgi,2018-07-30T02:35:44Z,554000,227871757,322,channels_must_6450_6499.csv,50.0,411.0,19.88,...,1079070.96,2457645.93,22384.80,423.72,0.025190,0.38,2247595.79,362878.32,180,320
198,UC8GVX05oeuw4vTWve7QYULw,가든의 세계여행,2014-10-31T09:03:12Z,183000,44870874,233,channels_must_4450_4499.csv,50.0,242.0,18.06,...,148513.44,50988.50,4294.90,367.82,0.033146,0.00,NaN,148513.44,183,317
12,UCd1IveTHngTF8ZyQ9f6uchw,공구왕황부장,2010-02-26T16:07:54Z,938000,365250537,922,channels_must_7150_7199.csv,50.0,3.0,1.86,...,138652.18,139375.84,1920.22,820.60,0.024065,0.08,106368.75,141459.43,183,317
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220,UCOtSwKUxIIP99lZfQ48Byag,화질덕후 4kOTAKU,2018-01-25T17:03:20Z,431000,274230653,1413,channels_must_6050_6099.csv,50.0,817.0,1.43,...,12797.50,6728.79,376.82,5.12,0.031754,0.00,NaN,12797.50,23,8
224,UClSxH8KQdfytf2raNAByMiA,Troom Troom Kr,2018-06-26T09:13:54Z,1000000,683071712,3236,channels_must_7200_7249.csv,50.0,6.0,2.00,...,9885.10,22613.27,61.96,2.28,0.008691,0.10,12867.80,9553.69,6,7
219,UCkjk77yUtLnuSalpGrlrG2Q,어바웃펫,2018-06-29T04:12:03Z,134000,46952823,608,channels_must_3850_3899.csv,50.0,319.0,4.27,...,884.84,1371.65,NaN,1.18,NaN,0.92,864.89,1114.25,3,5
211,UCbr6AL4ayehAkXcdBM68e8A,김PD전당포,2012-01-06T08:48:32Z,100000,2156709,135,channels_must_3250_3299.csv,50.0,448.0,36.69,...,1516.14,2263.11,34.86,2.16,0.015272,0.32,2111.50,1235.97,2,4
